## Setup

In [ ]:
import os
import time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd
from datetime import datetime
import io
from dotenv import load_dotenv

load_dotenv()

In [ ]:
BASE_URL = "http://jsoc.stanford.edu/cgi-bin/ajax/showextinfo"
SERIES = "hmi.sharp_720s"

METADATA = "T_REC,QUALITY"
SPACIAL_COORDS = "LON_FWT,LAT_FWT,LON_MIN,LON_MAX"
SHARP_FEATURES = (
    "USFLUX,MEANJZH,TOTUSJH,ABSNJZH,SAVNCPP,MEANPOT,TOTPOT,MEANALP,"
    "SHRGT45,TOTUSJZ,MEANSHR,MEANJZD,MEANGAM,MEANGBT,MEANGBZ,MEANGBH,"
    "TOTFZ,TOTBSQ,TOTSYQ,R_VALUE,AREA_ACR"
)

KEYWORDS = f"{METADATA},{SPACIAL_COORDS},{SHARP_FEATURES}"

NOTIFY_EMAIL = "dudiferraz@gmail.com"

DATA_INICIO_GLOBAL = "2010.05.01_00:00:00_TAI"
DATA_FIM_GLOBAL = "2024.12.31_23:59:00_TAI"

SAVE_PATH = os.getenv("MAG_RAW_PATH")

## Scrape Functions

In [ ]:
def criar_sessao_resiliente():
    session = requests.Session()

    estrategia_retry = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )

    adapter = HTTPAdapter(max_retries=estrategia_retry)
    session.mount("http://", adapter)
    session.mount("https://", adapter)

    return session

def gerar_lotes_mensais(start_str, end_str):
    formato_data = "%Y.%m.%d_%H:%M:%S"

    start_dt = datetime.strptime(start_str.replace('_TAI', ''), formato_data)
    end_dt = datetime.strptime(end_str.replace('_TAI', ''), formato_data)

    lotes = []
    current_start = start_dt

    while current_start < end_dt:
        current_end = current_start + pd.DateOffset(months=1)

        if current_end > end_dt:
            current_end = end_dt

        chunk_start = current_start.strftime(formato_data) + "_TAI"
        chunk_end = current_end.strftime(formato_data) + "_TAI"

        lotes.append((chunk_start, chunk_end))
        current_start = current_end

    return lotes

def extrair_dados_jsoc():
    if not SAVE_PATH:
        print("ERRO: SAVE_PATH não definido. Verifique seu arquivo .env")
        return

    os.makedirs(SAVE_PATH, exist_ok=True)

    http_session = criar_sessao_resiliente()

    lotes_tempo = gerar_lotes_mensais(DATA_INICIO_GLOBAL, DATA_FIM_GLOBAL)
    total_lotes = len(lotes_tempo)

    print(f"Iniciando extração resiliente. Total de lotes a processar: {total_lotes}\n")

    for indice, (start_chunk, end_chunk) in enumerate(lotes_tempo, start=1):
        print(f"[{indice}/{total_lotes}] Processando lote: {start_chunk} até {end_chunk}...")

        dataset_query = f"{SERIES}[][{start_chunk}-{end_chunk}]"

        params = {
            'ds': dataset_query,
            'key': KEYWORDS,
            'notify': NOTIFY_EMAIL,
            'seg': '',
            'dbhost': 'hmidb2',
            't': '1',
            'i': '1',
            'K': '0',
            'n': '0',
            'a': '0',
            'A': '0',
            'P': '0',
            'r': '0',
            'S': '0',
            'z': '0',
            'o': '0',
            'R': '0',
            'x': '0'
        }

        try:
            response = http_session.get(BASE_URL, params=params, timeout=60)
            print(f"   -> URL Consultada: {response.url}")
            response.raise_for_status()

            linhas_limpas = []
            cabecalho_encontrado = False

            for linha in response.text.split('\n'):
                if not linha.strip():
                    continue

                if linha.startswith('query\t'):
                    linha_formatada = linha.replace('query\t', 'DATASET_QUERY\t', 1)
                    linhas_limpas.append(linha_formatada)
                    cabecalho_encontrado = True

                elif cabecalho_encontrado and not (linha.startswith('string\t') or linha.startswith('%')):
                    linhas_limpas.append(linha)

            if len(linhas_limpas) > 1:
                texto_limpo = '\n'.join(linhas_limpas)
                df = pd.read_csv(io.StringIO(texto_limpo), sep='\t')

                ano_diretorio = start_chunk[:4]
                caminho_particionado = os.path.join(SAVE_PATH, ano_diretorio)

                os.makedirs(caminho_particionado, exist_ok=True)

                safe_start = start_chunk.replace(":", "").replace(".", "")
                safe_end = end_chunk.replace(":", "").replace(".", "")
                filename = f"jsoc_data_{safe_start}_to_{safe_end}.csv"

                filepath = os.path.join(caminho_particionado, filename)

                df.to_csv(filepath, index=False)
                print(f"   -> Sucesso! Dados salvos em: {ano_diretorio}/{filename} ({len(df)} linhas).")
            else:
                print("   -> Lote concluído, mas nenhum dado útil retornado (apenas metadados ou vazio).")

        except requests.exceptions.RetryError as e:
            print(f"   -> FALHA CRÍTICA: Lote abandonado após 5 tentativas de reconexão. Erro: {e}")
        except requests.exceptions.RequestException as e:
            print(f"   -> ERRO de conexão/HTTP ao baixar o lote: {e}")
        except Exception as e:
            print(f"   -> ERRO inesperado ao processar o lote: {e}")

        if indice < total_lotes:
            tempo_espera = 5
            time.sleep(tempo_espera)

    print("\nExtração finalizada com sucesso!")

## Main

In [ ]:
extrair_dados_jsoc()